# Map annotations to the full 6k comb object

NOTE: I tried in aws, I did not have enough RAM

## Rationale

Maintaining a full 6K-gene `comb` object ensures access to the complete CosMx panel rather than restricting downstream analyses to the top 2,000 highly variable genes. This is important for analyses that benefit from broader gene coverage, including spatial niche identification with NOVAE, pseudobulk differential expression with DESeq2, and ligand–receptor interaction analysis with CellChat.

## Code

Imports

In [1]:
# %% Libraries

import os
import scanpy as sc
import numpy as np
from pathlib import Path
import rapids_singlecell as rsc
import pandas as pd
from scipy.sparse import issparse

# %% Setting Paths
MAIN_DIR_NAME = "cosmx_gray"
MAIN_DIR = next(p for p in Path.cwd().parents if (p / MAIN_DIR_NAME).exists()) / MAIN_DIR_NAME
os.chdir(MAIN_DIR)

# %% Setting Seed
SEED_VALUE = 42
# set NumPy RNG for consistency
np.random.seed(SEED_VALUE)

# %% object versions
COMB_V= 'refined-clustered'
COMB_PATH = MAIN_DIR / 'data' / 'comb' / 'h5ad' / f'comb-{COMB_V}.h5ad'

# unintegrated is the last 6k version after QC
UNINTEGRATED_COMB_PATH = MAIN_DIR / 'data' / 'comb' / 'h5ad' / 'comb.h5ad'

# new object with 6k genes and annotations
FULL_COMB_V = 'annotated'
FULL_COMB_PATH = MAIN_DIR / 'data' / 'comb-full' / 'h5ad' / f'comb-full-{FULL_COMB_V}.h5ad'

/data/miniforge3/envs/rapids_singlecell/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# load comb obj
comb = sc.read_h5ad(COMB_PATH)


In [3]:
# get hgvs from comb
hgvs = comb.var.index.tolist()

In [4]:
# load airway epi obj
comb_full = sc.read_h5ad(UNINTEGRATED_COMB_PATH)

In [5]:
# %% define X_spatial
comb_full.obsm["X_spatial"] = np.array(
    comb_full.obs[['CenterX_global_px', 'CenterY_global_px']]
)

# %% Normalization and log1p transformation
comb_full.layers['counts'] = comb_full.X.copy() # saving the raw counts in a layer
sc.pp.normalize_total(comb_full, target_sum=1e4)
sc.pp.log1p(comb_full)
comb_full.raw = comb_full # this is just a snapshot of normalized+1logp counts. Used by default for plots.


In [6]:

# subset comb_full to only hgvs genes
comb_full = comb_full[:, hgvs]
comb_full

View of AnnData object with n_obs × n_vars = 1345908 × 2000
    obs: 'fov', 'Area', 'AspectRatio', 'Width', 'Height', 'Mean.PanCK', 'Max.PanCK', 'Mean.CD68', 'Max.CD68', 'Mean.Membrane', 'Max.Membrane', 'Mean.CD45', 'Max.CD45', 'Mean.DAPI', 'Max.DAPI', 'SplitRatioToLocal', 'NucArea', 'NucAspectRatio', 'Circularity', 'Eccentricity', 'Perimeter', 'Solidity', 'assay_type', 'version', 'Run_Tissue_name', 'Panel', 'cellSegmentationSetId', 'cellSegmentationSetName', 'slide_ID', 'CenterX_global_px', 'CenterY_global_px', 'unassignedTranscripts', 'median_RNA', 'RNA_quantile_0.75', 'RNA_quantile_0.8', 'RNA_quantile_0.85', 'RNA_quantile_0.9', 'RNA_quantile_0.95', 'RNA_quantile_0.99', 'nCount_RNA', 'nFeature_RNA', 'median_negprobes', 'negprobes_quantile_0.75', 'negprobes_quantile_0.8', 'negprobes_quantile_0.85', 'negprobes_quantile_0.9', 'negprobes_quantile_0.95', 'negprobes_quantile_0.99', 'nCount_negprobes', 'nFeature_negprobes', 'median_falsecode', 'falsecode_quantile_0.75', 'falsecode_quantile_

In [7]:
import gc
gc.collect()

127794

## Load model

In [8]:
import scvi

# %% load resolvi model
scvi_model_path = MAIN_DIR / 'data/comb' / 'resolvi_model'
model = scvi.external.RESOLVI.load(
    scvi_model_path, 
    adata=comb,
)

INFO     File /data/cosmx_gray/data/comb/resolvi_model/model.pt already         
         downloaded                                                             
RAPIDS SingleCell is installed and can be imported
RAPIDS SingleCell is installed and can be imported
RAPIDS SingleCell is installed and can be imported
RAPIDS SingleCell is installed and can be imported


/data/miniforge3/envs/rapids_singlecell/lib/python3.14/site-packages/scvi/data/fields/_dataframe_field.py:227: UserWarning: Category 0 in adata.obs['_scvi_ind_x'] has fewer than 3 cells. Models may not train properly.
  new_mapping = _make_column_categorical(
/data/miniforge3/envs/rapids_singlecell/lib/python3.14/site-packages/scvi/model/base/_save_load.py:158: FutureWarning: RESOLVI is a spatial transcriptomics model that will be moved to the scvi-tools spatial companion package `scviva-tools` starting in scvi-tools v1.5 and will no longer be supported here. It will be deprecated from scvi-tools in v1.6.
  model = cls(adata, **non_kwargs, **kwargs)
INFO: GPU available: True (cuda), used: True
2026-07-19 14:14:12 | [INFO] GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
2026-07-19 14:14:12 | [INFO] TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/pro

Epoch 1/50:   0%|          | 0/50 [00:00<?, ?it/s]

/data/miniforge3/envs/rapids_singlecell/lib/python3.14/site-packages/scvi/external/resolvi/_module.py:383: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:838.)
  concentration=torch.tensor(
2026-07-19 14:14:13 | [INFO] Guessed max_plate_nesting = 2


Epoch 1/50:   2%|▏         | 1/50 [00:00<00:18,  2.61it/s, v_num=1]

INFO: `Trainer.fit` stopped: `max_steps=1` reached.
2026-07-19 14:14:13 | [INFO] `Trainer.fit` stopped: `max_steps=1` reached.


Epoch 1/50:   2%|▏         | 1/50 [00:00<00:19,  2.51it/s, v_num=1]
RESOLVI Model with the following params: 
n_hidden: 32 n_latent: 10, n_layers: 2, dropout_rate: 0.05, dispersion: gene, 
gene_likelihood: nb n_neighbors: 10
Training status: Trained


## Query Transfer

In [9]:
comb_full.obs["predicted_celltype"] = "unknown"
comb_full.obs_names = [f"query_{i}" for i in comb_full.obs_names]

<positron-console-cell-10>:1: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.


RESOLVI Model with the following params: 
n_hidden: 32 n_latent: 10, n_layers: 2, dropout_rate: 0.05, dispersion: gene, 
gene_likelihood: nb n_neighbors: 10
Training status: Trained


In [11]:
sc.pp.filter_cells(comb, min_genes=5)
comb_full

In [10]:
model.prepare_query_anndata(comb_full, reference_model=model)
query_resolvi = model.load_query_data(comb_full, reference_model=model)

INFO     Found 100.0% reference vars in query data.                             


AssertionError: Please filter cells with less than 5 counts prior to running resolVI.

Just need to replace the metadata

Get all the cell names and annotations

Save the annotated object

In [ ]:
comb_full.write_h5ad(FULL_COMB_PATH)